# JSON Basics — 07: JSON Schema Validation

Validating incoming JSON against an expected schema is critical
for data quality in production pipelines.

Topics: schema validation with FAILFAST/PERMISSIVE, custom validators,
JSON Schema (Python jsonschema library), Spark column validators, quarantine pipeline.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/json_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('json-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Spark Mode-Based Validation



In [ ]:

import json as pyjson, pathlib

# Create JSON with various quality issues
bad_json = [
    '{"id":1,"amount":"not_a_number","status":"confirmed","date":"2024-01-15"}',
    '{"id":2,"amount":100.5,"status":"confirmed","date":"2024-01-16"}',
    '{"id":3,"amount":200.0}',  # missing fields
    'INVALID JSON HERE',
    '{"id":4,"amount":-50.0,"status":"refunded","date":"2024-01-18"}',  # valid
    '{"id":5,"amount":null,"status":null,"date":null}',  # nulls
]
bad_path = f"{DATA_DIR}/validate_test.json"
with open(bad_path,"w") as f: [f.write(r+"\n") for r in bad_json]

# PERMISSIVE — capture bad rows
from pyspark.sql.types import *
schema = StructType([
    StructField("id",     LongType()),
    StructField("amount", DoubleType()),
    StructField("status", StringType()),
    StructField("date",   DateType()),
    StructField("_corrupt_record", StringType()),
])
df_perm = spark.read.schema(schema).option("mode","PERMISSIVE") \
               .option("columnNameOfCorruptRecord","_corrupt_record").json(bad_path)
good = df_perm.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
bad  = df_perm.filter(F.col("_corrupt_record").isNotNull())
print(f"PERMISSIVE: {good.count()} good, {bad.count()} bad")
good.show(); bad.select("_corrupt_record").show(truncate=False)


## Step 2 — Business Rule Validation



In [ ]:

df = spark.read.json(f"{DATA_DIR}/orders.ndjson")

# Validate business rules
df_validated = df.withColumn("valid_price",     F.col("price") > 0) \
                 .withColumn("valid_quantity",   F.col("quantity").between(1,100)) \
                 .withColumn("valid_status",     F.col("status").isin("pending","confirmed","shipped","delivered","cancelled")) \
                 .withColumn("valid_date",       F.col("order_date").isNotNull()) \
                 .withColumn("_is_valid",        F.col("valid_price") & F.col("valid_quantity") & F.col("valid_status") & F.col("valid_date"))

valid   = df_validated.filter(F.col("_is_valid"))
invalid = df_validated.filter(~F.col("_is_valid"))
print(f"Valid rows  : {valid.count():,}")
print(f"Invalid rows: {invalid.count():,}")
if invalid.count() > 0:
    print("\nInvalid rows sample:")
    invalid.select("order_id","price","quantity","status","valid_price","valid_quantity","valid_status").show(5)


## Step 3 — Complete Validation Pipeline with Quarantine



In [ ]:

import datetime as dt

def validate_and_quarantine(input_path, good_output, bad_output, schema):
    schema_with_corrupt = StructType(schema.fields + [StructField("_corrupt_record",StringType())])
    df_raw = spark.read.schema(schema_with_corrupt) \
                  .option("mode","PERMISSIVE") \
                  .option("columnNameOfCorruptRecord","_corrupt_record") \
                  .json(input_path)
    df_good = df_raw.filter(F.col("_corrupt_record").isNull()) \
                    .drop("_corrupt_record") \
                    .filter(F.col("price") > 0) \
                    .filter(F.col("quantity").between(1,100)) \
                    .filter(F.col("status").isin("pending","confirmed","shipped","delivered","cancelled"))
    df_bad  = df_raw.filter(F.col("_corrupt_record").isNotNull()) \
                    .select("_corrupt_record",
                            F.current_timestamp().alias("_rejected_at"),
                            F.lit(str(input_path)).alias("_source"))
    df_good.write.mode("overwrite").parquet(good_output)
    df_bad.write.mode("overwrite").parquet(bad_output)
    return df_good.count(), df_bad.count()

from pyspark.sql.types import *
schema=StructType([StructField(k,t) for k,t in [
    ("order_id",LongType()),("customer_id",LongType()),("product",StringType()),
    ("category",StringType()),("country",StringType()),("quantity",IntegerType()),
    ("price",DoubleType()),("status",StringType()),("order_date",DateType())]])

good_count,bad_count = validate_and_quarantine(
    f"{DATA_DIR}/orders.ndjson",
    f"{DATA_DIR}/validated/good",
    f"{DATA_DIR}/validated/bad", schema)
print(f"Validation complete: {good_count:,} good, {bad_count:,} bad")


## Summary



In [ ]:

print("""
JSON validation strategy:
  1. Spark structural validation: PERMISSIVE + columnNameOfCorruptRecord
     → catches malformed JSON, wrong types
  
  2. Business rule validation: filter on column conditions
     → catches valid JSON with invalid values

  3. Quarantine pattern:
     good rows → write to Parquet (analytics zone)
     bad rows  → write to quarantine Parquet (for investigation)

Always validate at the ENTRY point of your pipeline.
Never assume JSON from external sources is clean.
""")
